# Qwen 3.5 Sales QLoRA on Google Colab
Chọn GPU runtime trước khi chạy. Upload file ZIP tạo bởi `scripts/package_for_colab.ps1` vào `MyDrive/sales-finetune/`. Notebook tạo lại 1.000.000 mẫu tại ổ local Colab và lưu checkpoint adapter vào Google Drive.

In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/sales-finetune')
ARCHIVE = DRIVE_ROOT / 'sales-finetune-colab.zip'
WORKSPACE = Path('/content/sales-finetune')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print(DRIVE_ROOT)

In [ ]:
import zipfile
if not ARCHIVE.is_file():
    raise FileNotFoundError(f'Upload ZIP to {ARCHIVE}')
WORKSPACE.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(ARCHIVE) as package:
    for member in package.infolist():
        destination = (WORKSPACE / member.filename).resolve()
        if WORKSPACE.resolve() not in destination.parents and destination != WORKSPACE.resolve():
            raise ValueError(f'Unsafe archive path: {member.filename}')
        package.extract(member, WORKSPACE)
print(WORKSPACE)

In [ ]:
%cd /content/sales-finetune
!pip install -q -r requirements-colab.txt
!pip install -q -e .
!nvidia-smi

In [ ]:
!python -m llm_finetune_gpt_oos.main generate-sales-playbook --config configs/synthetic_sales.yaml

In [ ]:
import yaml
config_path = WORKSPACE / 'configs' / 'train.yaml'
runtime_path = WORKSPACE / 'configs' / 'colab_runtime.yaml'
with config_path.open(encoding='utf-8') as handle:
    config = yaml.safe_load(handle)
artifact_dir = DRIVE_ROOT / 'artifacts' / 'qwen35-sales-qlora'
cache_dir = DRIVE_ROOT / 'huggingface-cache'
checkpoints = sorted(artifact_dir.glob('checkpoint-*'), key=lambda item: int(item.name.rsplit('-', 1)[-1])) if artifact_dir.exists() else []
config['model']['cache_dir'] = str(cache_dir)
config['model']['quantization']['compute_dtype'] = 'float16'
config['training']['output_dir'] = str(artifact_dir)
config['training']['max_seq_length'] = 1024
config['training']['bf16'] = False
config['training']['fp16'] = True
config['training']['max_steps'] = 10000
config['training']['resume_from_checkpoint'] = str(checkpoints[-1]) if checkpoints else None
with runtime_path.open('w', encoding='utf-8') as handle:
    yaml.safe_dump(config, handle, allow_unicode=True, sort_keys=False)
print(runtime_path)
print(config['training']['resume_from_checkpoint'])

In [ ]:
!python -m llm_finetune_gpt_oos.main train --config configs/colab_runtime.yaml

In [ ]:
!python -m llm_finetune_gpt_oos.main evaluate --config configs/colab_runtime.yaml
!ls -lah /content/drive/MyDrive/sales-finetune/artifacts/qwen35-sales-qlora